# P04｜像素腦讀論文：ResNet

論文：Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun, "Deep Residual Learning for Image Recognition", CVPR 2016。https://arxiv.org/abs/1512.03385

這本筆記本有兩部分：
1. 用開源的 ResNet-50 看貓咪照片：ONNX Model Zoo 的 resnet50-v1-7（用 MXNet／Gluon 在 ImageNet 訓練、轉成 ONNX，Apache 2.0）。這不是論文作者發表的原始模型（作者發表的是 Caffe 模型：https://github.com/KaimingHe/deep-residual-networks ）。如果模型網址下載失敗，可以改從 Hugging Face 的 onnxmodelzoo 下載同一個檔案。
2. **像素腦的示意版**：照論文 CIFAR-10 實驗的網路形狀（6n + 2 層、16／32／64 個通道、捷徑用選項 A），在 scikit-learn 內附的 8×8 手寫數字上，比較一般網路和殘差網路。這是示意，不是論文的程式，資料也不是 CIFAR-10。

全部用 CPU 就能跑。

## 0. 準備：安裝套件、下載兩張貓咪照片（都是 CC0：一隻貓 AdinaVoicu、兩隻小貓 Irshath 25，Wikimedia Commons）和 ResNet-50 模型（約 100 MB）

In [ ]:
!pip -q install onnxruntime

In [ ]:
import os, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from PIL import Image

for url, f in [("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep02-edges/cat.png", "cat.png"), ("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/p02-yolo/cats.jpg", "cats.jpg"),
               ("https://media.githubusercontent.com/media/onnx/models/main/validated/vision/classification/resnet/model/resnet50-v1-7.onnx", "resnet50.onnx"), ("https://raw.githubusercontent.com/onnx/models/main/validated/vision/classification/synset.txt", "synset.txt")]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(url, f)

## 1. 先看成果：ResNet-50 看貓咪照片

輸出 ImageNet 1000 類裡機率最高的 3 類。

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession("resnet50.onnx")      # 開源的 ResNet-50（ONNX Model Zoo），不是論文作者的原始模型
names = [l.split(" ", 1)[1].split(",")[0].strip() for l in open("synset.txt")]

def classify(img):
    img = img.convert("RGB")
    s = 256 / min(img.size)                         # 短邊縮到 256，再切中間 224×224
    img = img.resize((round(img.width * s), round(img.height * s)))
    l, t = (img.width - 224) // 2, (img.height - 224) // 2
    img = img.crop((l, t, l + 224, t + 224))
    x = (np.asarray(img, np.float32) / 255 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    x = x.transpose(2, 0, 1)[None].astype(np.float32)
    logits = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]
    p = np.exp(logits - logits.max()); p /= p.sum()
    return [(names[i], p[i]) for i in p.argsort()[::-1][:3]]

for f in ["cat.png", "cats.jpg"]:
    t0 = time.time(); top = classify(Image.open(f))
    print(f, f"{(time.time() - t0) * 1000:.0f} 毫秒")
    for n, p in top:
        print(f"  {n} {p:.0%}")

## 2. 換個方式再看一次：上下顛倒、兩隻小貓、一隻一隻切開

In [ ]:
cat, cats = Image.open("cat.png"), Image.open("cats.jpg")
tests = {"一隻貓": cat,
         "上下顛倒": cat.rotate(180),
         "兩隻小貓": cats,
         "只切左邊那隻": cats.crop((280, 200, 1000, 1150)),
         "只切右邊那隻": cats.crop((960, 230, 1800, 1120))}
for k, im in tests.items():
    n, p = classify(im)[0]
    print(f"{k}：{n} {p:.0%}")

## 3. 示意版：資料

In [ ]:
X, y = load_digits(return_X_y=True)             # 1797 張 8×8 手寫數字
X = torch.tensor(X / 16, dtype=torch.float32).reshape(-1, 1, 8, 8)
y = torch.tensor(y)
print(X.shape, "10 類")

## 4. 網路：一般網路和殘差網路只差一個加號

`shortcut=True` 時，每兩層卷積的輸出再加上輸入 x。

In [ ]:
class Block(nn.Module):                         # 兩層 3×3 卷積
    def __init__(self, cin, cout, stride, shortcut):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout))
        self.shortcut = shortcut
        self.down = (cin != cout)                   # 換尺寸時，捷徑縮小並補 0（論文的選項 A）
    def forward(self, x):
        out = self.f(x)
        if self.shortcut:                           # 殘差：輸出 = F(x) + x
            if self.down:
                x = x[:, :, ::2, ::2]
                x = torch.cat([x, torch.zeros_like(x)], 1)
            out = out + x
        return torch.relu(out)

def make_net(n, shortcut):                          # 論文 CIFAR 版：6n + 2 層
    layers = [nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU()]
    cin = 16
    for cout, stride in [(16, 1), (32, 2), (64, 2)]:
        for i in range(n):
            layers.append(Block(cin, cout, stride if i == 0 else 1, shortcut))
            cin = cout
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)]
    return nn.Sequential(*layers)

for n in [3, 9]:
    print(f"{6 * n + 2} 層：", sum(p.numel() for p in make_net(n, False).parameters()), "個參數")

## 5. 訓練四個網路，看「訓練資料」答錯多少

論文的重點：一般網路變深，連訓練資料都答得比較差（不是過度擬合）。每個網路都用一樣的設定、一樣的亂數種子。在 Colab 的 CPU 上大約要幾分鐘，數字可能會跟影片有一點點不同。

In [ ]:
def train(n, shortcut, epochs=30, seed=0):
    torch.manual_seed(seed)
    net = make_net(n, shortcut)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, [20, 25], 0.1)   # 跟論文一樣，後段學習率除以 10
    errs = []
    for ep in range(epochs):
        net.train()
        for idx in torch.randperm(len(X)).split(128):
            loss = nn.functional.cross_entropy(net(X[idx]), y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        net.eval()
        with torch.no_grad():                       # 看「訓練資料」答錯幾 %
            errs.append((net(X).argmax(1) != y).float().mean().item() * 100)
    return net, errs

curves = {}
for name, n, sc in [("一般 20 層", 3, False), ("一般 56 層", 9, False),
                    ("殘差 20 層", 3, True), ("殘差 56 層", 9, True)]:
    t0 = time.time()
    _, curves[name] = train(n, sc)
    print(f"{name}：第 5 輪錯 {curves[name][4]:.1f}%，第 30 輪錯 {curves[name][-1]:.1f}%，{time.time() - t0:.0f} 秒")

In [ ]:
for name, e in curves.items():
    label = ("plain-" if "一般" in name else "ResNet-") + name[3:5]
    plt.plot(range(1, 31), e, "--" if "一般" in name else "-", label=label)
plt.xlabel("epoch"); plt.ylabel("training error %"); plt.legend(); plt.show()

## 6. 咦？是梯度消失嗎？

論文說：有 BatchNorm 的一般網路，梯度並沒有消失。我們也量量看。

In [ ]:
def first_grad(n, shortcut):                    # 剛開始訓練時，第一層拿到的梯度有多大
    torch.manual_seed(0)
    net = make_net(n, shortcut)
    nn.functional.cross_entropy(net(X[:128]), y[:128]).backward()
    return net[0].weight.grad.norm().item()

for n in [3, 9]:
    print(f"一般 {6 * n + 2} 層：第一層梯度 {first_grad(n, False):.3f}")

## 7. 為什麼殘差比較好學：什麼都不做，只要把 F(x) 歸零

In [ ]:
blk = Block(16, 16, 1, shortcut=True)
nn.init.zeros_(blk.f[4].weight)                     # 把 F(x) 最後一層歸零
x = torch.rand(1, 16, 8, 8)
print("殘差區塊：輸出等於輸入？", torch.allclose(blk(x), x))
plain = Block(16, 16, 1, shortcut=False)
nn.init.zeros_(plain.f[4].weight)
print("一般區塊：輸出等於輸入？", torch.allclose(plain(x), x), "（全部變成", plain(x).abs().max().item(), "）")

## 🧪 自己玩玩看
- 把 `train` 的 `n` 改成 18（110 層），一般網路和殘差網路差多少？
- 把學習率 `lr` 改成 0.01，一般 56 層會不會比較好？
- 第 1 步換成你自己的照片試試看。

像素腦 @pixelbraintw